# 01 — Exploratory Data Analysis: InfraredSolarModules

Covers:
1. Load metadata and summarise class distribution
2. Bar chart of class counts
3. Image grid — 5 random samples per class
4. Image dimension consistency check
5. W&B artifact logging

In [ ]:
import json
import random
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Works whether the notebook is run from project root or notebooks/
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "InfraredSolarModules" / "InfraredSolarModules"
META_PATH = DATA_DIR / "module_metadata.json"

assert META_PATH.exists(), f"Metadata not found at {META_PATH}"
print(f"Project root : {PROJECT_ROOT}")
print(f"Data dir     : {DATA_DIR}")

## 1. Load metadata

In [ ]:
with open(META_PATH) as f:
    metadata = json.load(f)

records = [
    {"id": k, "path": DATA_DIR / v["image_filepath"], "cls": v["anomaly_class"]}
    for k, v in metadata.items()
]

class_counts = Counter(r["cls"] for r in records)
# Sort by count descending for consistent ordering throughout notebook
CLASSES = sorted(class_counts, key=lambda c: -class_counts[c])

print(f"Total images : {len(records)}\n")
print(f"{'Class':<25} {'Count':>6}  {'Share':>6}")
print("-" * 40)
for cls in CLASSES:
    n = class_counts[cls]
    print(f"{cls:<25} {n:>6}  {n/len(records)*100:>5.1f}%")

## 2. Class distribution

In [ ]:
counts = [class_counts[c] for c in CLASSES]
colors = ["#FF7043" if c == "No-Anomaly" else "#1976D2" for c in CLASSES]

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(CLASSES, counts, color=colors, edgecolor="white", linewidth=0.5)
ax.set_ylabel("Number of images")
ax.set_title("InfraredSolarModules — class distribution  (n = 20 000)", pad=12)
ax.tick_params(axis="x", rotation=40)
ax.set_ylim(0, max(counts) * 1.12)

for bar, n in zip(bars, counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 80,
        str(n), ha="center", va="bottom", fontsize=8.5
    )

# Imbalance annotation
majority, minority = max(counts), min(counts)
ax.annotate(
    f"Imbalance ratio: {majority/minority:.0f}× (No-Anomaly vs {CLASSES[counts.index(minority)]})",
    xy=(0.98, 0.97), xycoords="axes fraction", ha="right", va="top",
    fontsize=9, color="#555"
)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "notebooks" / "class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Image grid — 5 random samples per class

In [ ]:
random.seed(42)
by_class = {cls: [r for r in records if r["cls"] == cls] for cls in CLASSES}

N_SAMPLES = 5
n_classes = len(CLASSES)

fig, axes = plt.subplots(
    n_classes, N_SAMPLES,
    figsize=(N_SAMPLES * 2.2, n_classes * 2.2),
    gridspec_kw={"hspace": 0.05, "wspace": 0.03}
)

for row, cls in enumerate(CLASSES):
    samples = random.sample(by_class[cls], N_SAMPLES)
    for col, rec in enumerate(samples):
        img = Image.open(rec["path"]).convert("L")
        axes[row, col].imshow(img, cmap="inferno", aspect="auto")
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])
        if col == 0:
            axes[row, col].set_ylabel(cls, fontsize=8, rotation=0,
                                      labelpad=90, va="center")

fig.suptitle("5 random thermal images per class", y=1.005, fontsize=12)
plt.savefig(PROJECT_ROOT / "notebooks" / "image_grid.png", dpi=120, bbox_inches="tight")
plt.show()

## 4. Image dimension analysis

In [ ]:
# PIL reads only the image header to get size — fast even for 20k images
sizes = []
for rec in records:
    with Image.open(rec["path"]) as img:
        sizes.append(img.size)  # (W, H)

widths  = np.array([s[0] for s in sizes])
heights = np.array([s[1] for s in sizes])
unique_sizes = Counter(sizes)

print(f"Unique (W×H) combinations : {len(unique_sizes)}")
print(f"Width  — min {widths.min()}, max {widths.max()}, mean {widths.mean():.1f}, std {widths.std():.1f}")
print(f"Height — min {heights.min()}, max {heights.max()}, mean {heights.mean():.1f}, std {heights.std():.1f}")
print(f"\nTop sizes:")
for (w, h), cnt in unique_sizes.most_common(10):
    print(f"  {w:>4}×{h:<4}  {cnt:>6} images  ({cnt/len(records)*100:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, vals, label in zip(axes, [widths, heights], ["Width (px)", "Height (px)"]):
    ax.hist(vals, bins=40, color="#1976D2", edgecolor="white", linewidth=0.4)
    ax.set_xlabel(label)
    ax.set_ylabel("Count")
    ax.set_title(f"Distribution of image {label.split()[0].lower()}s")
    ax.axvline(int(np.median(vals)), color="#FF7043", lw=1.5,
               label=f"median = {int(np.median(vals))}px")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "notebooks" / "image_dimensions.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. W&B — log class distribution as artifact

Run `wandb login` once before executing this cell.

In [ ]:
import wandb

try:
    run = wandb.init(project="solar-thermal-cv", name="eda", job_type="eda")

    table = wandb.Table(columns=["class", "count", "share_pct"])
    for cls in CLASSES:
        n = class_counts[cls]
        table.add_data(cls, n, round(n / len(records) * 100, 2))
    run.log({"class_distribution": table})

    artifact = wandb.Artifact("eda-figures", type="figures")
    for fig_path in (PROJECT_ROOT / "notebooks").glob("*.png"):
        artifact.add_file(str(fig_path))
    run.log_artifact(artifact)

    run.finish()
    print("Logged to W&B.")
except wandb.errors.UsageError:
    print("W&B not authenticated — run `wandb login` then re-execute this cell.")